# Smoke Test: Single Time Step, No Controllers

This notebook runs a smoke test for the Pandapipes time-series module using a single time step and no controllers.


In [5]:
import pandapipes as pp
from pandapipes.timeseries import run_timeseries
from pandapipes.pipeflow import pipeflow as _original_pipeflow

# Monkey-patch pipeflow to count calls
call_count = {'count': 0}

def counting_pipeflow(net, *args, **kwargs):
    call_count['count'] += 1
    return _original_pipeflow(net, *args, **kwargs)

# Build a simple network: 2 junctions, 1 ext_grid, 1 pipe
net = pp.create_empty_network(fluid='water')
j1 = pp.create_junction(net, pn_bar=1.0, tfluid_k=293.15)
j2 = pp.create_junction(net, pn_bar=1.0, tfluid_k=293.15)
pp.create_ext_grid(net, junction=j1, p_bar=1.0, t_k=293.15, mdot_kg_per_s=1.0)
pp.create_pipe_from_parameters(net,
    from_junction=j1, to_junction=j2,
    length_km=0.01, diameter_m=0.1, k_mm=0.1)

# Run single-step time series
run_timeseries(net, time_steps=[0], run=counting_pipeflow)

# Assertions to verify behavior
assert call_count['count'] == 1, f"Expected pipeflow to be called once, but got {call_count['count']}"
assert hasattr(net, 'res_pipe'), 'Result table res_pipe not created'
assert hasattr(net, 'res_junction'), 'Result table res_junction not created'

print('Smoke test passed: pipeflow called once and result tables exist.')

100%|██████████| 1/1 [00:00<00:00, 58.79it/s]

Smoke test passed: pipeflow called once and result tables exist.
